<font color="#0D9DF0"><h3><b>ANÁLISE DO DESENVOLVIMENTO HUMANO E ECONÔMICO GLOBAL</b></h3></font>

**Base de dados:** Gapminder  
**Grupo:** [preencher nomes]  
**Etapa atual:** Planejamento

Objetivo do trabalho
Responder as seguintes questoes tendo com dado base inicial o gapminder disponibilizado pelo professor

1- Países que gastam mais em saúde per capita têm maior expectativa de vida, mesmo quando o PIB per capita é parecido?

2- A mortalidade infantil explica melhor as diferenças de expectativa de    vida entre países ricos e pobres do que o PIB per capita?

3 - Países com mais veículos por habitante têm melhor longevidade?

4 - A taxa de obesidade cresce junto com o PIB per capita?

5 - Países com mais médicos por mil habitantes têm maior expectativa de
vida, mesmo descontando o efeito do PIB per capita?





Etapas do notebook
1. Coleta e exploração inicial
2. Limpeza e preparação
3. Análise exploratória
4. Funções de visualização
5. Interpretação das análises
6. Síntese e conclusão

#### Importação das bibliotecas

In [8]:
# Importar as bibliotecas que serão utilizadas pelo grupo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import csv
from charset_normalizer import from_bytes #Para detectar encoding e separador dos arquivos automaticamente
import urllib.request as ur   #Para aceitar base de dados que venham de URL

---

## 1. Coleta de Dados
**Etapa atual:  COLETA**

O primeiro passo é carregar a base Gapminder e registrar sua origem. Confiram se o arquivo foi disponibilizado junto com o trabalho e anotem a fonte utilizada.

**Links ou caminhos bases de dados**

In [9]:
# Substitua a URL abaixo pela URL 'raw' do seu arquivo no GitHub
url_github = 'https://raw.githubusercontent.com/AndremRb/Analise-do-Desenvolvimento-Humano-e-Global/refs/heads/main/gapminder_full.csv'


**Função para descobrir o encoding e o separador do arquivo**


In [10]:
def configuraçoes_csv(caminho_arquivo):
    """
    Descobre o encoding e o separador de qualquer CSV de forma rápida e certeira.
    Aceita tanto arquivos locais quanto links da internet (ex: GitHub).
    Retorna: (encoding, separador).
    """

    # 1. Verifica se é um link da internet ou arquivo local
    if caminho_arquivo.startswith('http://') or caminho_arquivo.startswith('https://'):   # Verifica se começa com um dos termos
        # Se for link, faz uma requisição HTTP e lê os primeiros 20.000 bytes
        req = ur.Request(
            caminho_arquivo,
            headers={'User-Agent': 'Mozilla/5.0'} # Simula um cabeçalho real na requisição como se fosse um usuário acessando pelo Mozila
        )
        with ur.urlopen(req) as f:   # Abre a conexão com o servidor através da requisição anterior criada
            amostra_binaria = f.read(20000)   #Lê e armazena os primeiros 20000 bytes em amostra_binária
                                              #NOTA: não usamos open(caminho_arquivo, "rb") pq na internet tudo já está em byte
    else:
        # Se for arquivo local, abre normalmente
        with open(caminho_arquivo, 'rb') as f:    #Aqui usamos o 'rb' pois arquivos locais não costuman estar em binário pois nós não conseguiriamos ler.
            amostra_binaria = f.read(20000)

    # 2. Descobre o encoding com quase 100% de precisão
    resultado = from_bytes(amostra_binaria).best()  # Analisa a amostra binária
    encoding = resultado.encoding   # Armazena o nome do encoding detectado
    texto_amostra = str(resultado)    # Converte em texto legível novamente

    # 3. Descobre o separador (.csv, .txt, etc)
    dialeto = csv.Sniffer().sniff(texto_amostra)    #"Fareja" em nossa amostra qual o separador mais aparece e
    separador = dialeto.delimiter

    return caminho_arquivo,encoding, separador

**Função para criar o Data Frame**

In [11]:
def cria_dataframe(caminho_arquivo,encoding,separador):
    df=pd.read_csv(caminho_arquivo, encoding=encoding, sep=separador)
    return df


**Função para carregar e criar multiplos Data Frame**

In [60]:
def carrega_multiplos_dataframes(lista_caminhos):
    """
    Recebe uma lista de URLs ou caminhos locais e retorna uma lista de DataFrames carregados
    automaticamente usando as configurações detectadas para cada um deles.

    **Exemplo de entrada:**
    lista caminhos= [url1, url2, url3, ...] ou [caminho1, caminho2, caminho3, ...]

    **Exemplo de saída:**
    dataframes= [df1, df2, df3, ...]
    """
    dataframes = []   #cria uma lista vazia para nosso append ir add oq precisa
    for caminho in lista_caminhos:
        # 1. Detecta as configurações do arquivo atual
        config = configuraçoes_csv(caminho)
        # 2. Cria o DataFrame correspondente
        df = cria_dataframe(*config)#garante que cada termo da tupla seja descompactado e utilizado na função
        # 3. Adiciona o DataFrame à lista
        dataframes.append(df)
    return dataframes

# Exemplo de uso prático:
# lista_de_fontes = [url_github]  # Você pode colocar quantos links ou caminhos quiser aqui
# meus_dfs = carregar_multiplos_dataframes(lista_de_fontes)
# meus_dfs[0].head()  # Acessa o primeiro DataFrame gerado

In [45]:
df_pib = carrega_multiplos_dataframes({url_github})[0] #Escolha qual dataframe quer ver da nossa lista pelo indice

## 2. Exploração Inicial dos Dados



**Função para criar lista de informações de multiplas tabelas**

---



In [55]:
def informaçoes_df(lista_dataframes):
  """Informa o range do index,numero de colunas,o tipo de dado de
  cada coluna e a quantidade de linhas NÃO NULAS de cada dataframe.

  **Exemplo de entrada:**
  lista_dataframes= [df1, df2, df3, ...]
 """
  dfs_informaçoes = []
  for df in lista_dataframes:
    dfs_informaçoes.append(df.info()) #Adiciona ao fim da lista a tabrla com as informaçoes do df a cada nova passada
  return dfs_informaçoes


**Função para criar lista de resumo estátistico de multiplas tabelas**



In [57]:
def multiplos_resumos_estatisticos(lista_dataframes):
  """Traz um resumo estatístico de cada Data Frame e armazena em uma lista cada resumo

  **Exemplo de entrada:**
  lista_dataframes= [df1, df2, df3, ...]
 """
  dfs_resumos_estatisticos = []
  for df in lista_dataframes:
    dfs_resumos_estatisticoss.append(df.describe().round(2)) #Adiciona ao fim da lista a tabela com as informaçoes do df a cada nova passada
  return dfs_resumos_estatisticos


In [58]:
# Dicionário global que armazenará todas as nossas bases e seus metadados de forma automatizada
dados_projeto = {}

def cadastrar_fontes_de_dados(lista_fontes):
    """
    Carrega automaticamente os dataframes de uma lista de caminhos/URLs
    e estrutura cada um deles com seus respectivos metadados dentro do dicionário global.
    """
    global dados_projeto

    for i, caminho in enumerate(lista_fontes):
        # 1. Detecta configurações e cria o DataFrame
        config = configuraçoes_csv(caminho)
        df = cria_dataframe(*config)

        # 2. Define um nome identificador automático para a base
        nome_base = caminho.split('/')[-1].replace('.csv', '').replace('_full', '')
        if not nome_base:
            nome_base = f"base_{i+1}"

        # 3. Estrutura o DataFrame junto com seus metadados automaticamente
        dados_projeto[nome_base] = {
            'df': df,
            'origem': caminho,
            'linhas': df.shape[0],
            'colunas': list(df.columns),
            'resumo_estatistico': df.describe().round(2)
        }

    print(f"Sucesso! {len(lista_fontes)} base(s) carregada(s) e estruturada(s) no dicionário 'dados_projeto'.")

# Exemplo de Uso Automatizado:
# Cadastra a sua fonte atual do GitHub
fontes = [url_github]
cadastrar_fontes_de_dados(fontes)

# Para visualizar as informações cadastradas de forma limpa:
print("\nBases cadastradas:", list(dados_projeto.keys()))
print("Quantidade de linhas da base gapminder:", dados_projeto['gapminder']['linhas'])
display(dados_projeto['gapminder']['resumo_estatistico'])

Sucesso! 1 base(s) carregada(s) e estruturada(s) no dicionário 'dados_projeto'.

Bases cadastradas: ['gapminder']
Quantidade de linhas da base gapminder: 1736


,year,pop,lifeExp,gdpPercap
count,1736.00,1.736000e+03,1736.00,1736.00
mean,1979.67,3.156800e+07,59.51,7240.16
std,17.30,1.172888e+08,12.96,9839.87
min,1952.00,6.001100e+04,23.60,241.17
25%,1967.00,2.818694e+06,48.24,1202.06
50%,1982.00,7.150606e+06,60.81,3534.77
75%,1997.00,1.977102e+07,70.94,9380.25
max,2007.00,1.318683e+09,82.60,113523.13


**Anotações do grupo**
- Quais colunas estão disponíveis?
- Qual é o período coberto pelos dados?
- Há algo que precisa ser verificado antes da limpeza?

## 3. Limpeza de Dados
**Etapa atual:  TRATAMENTO**

A atividade solicita excluir os dados ausentes e eliminar registros duplicados. Antes e depois da limpeza, registrem a quantidade de linhas e as decisões tomadas.

In [16]:
# Verificar a quantidade de valores ausentes por coluna

In [17]:
# Verificar a quantidade de registros duplicados

In [18]:
# Remover os valores ausentes conforme solicitado no enunciado

In [19]:
# Eliminar registros duplicados

In [20]:
# Conferir a base após a limpeza

**Decisões de tratamento**
- Quantos valores ausentes foram encontrados?
- Quantas linhas duplicadas foram removidas?
- Quantos registros restaram?
- Foi necessário ajustar algum tipo de dado? Justifique.

## 4. Transformação e Preparação dos Dados
**Etapa atual:  PREPARAÇÃO**

Avaliem se é necessário ajustar tipos de dados, nomes de colunas ou criar agrupamentos para facilitar as análises. Registrem apenas as transformações que realmente forem utilizadas.

In [21]:
# Realizar aqui as transformações necessárias, se houver

## 5. Análise Exploratória de Dados (AED)
**Etapa atual: 🟩 ANÁLISE**

Escolham pelo menos **15 perguntas** para investigar. Para cada análise, incluam código, tabela ou gráfico e uma interpretação escrita pelo grupo. As perguntas abaixo são sugestões; vocês podem adaptar ou substituir por outras relacionadas ao objetivo do trabalho.

### Análise 1 — Evolução da expectativa de vida
**Pergunta:** Como a expectativa de vida global mudou entre 1952 e 2007?

**Visualização possível:** Gráfico de linhas ao longo dos anos.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [22]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 2 — Evolução do PIB per capita
**Pergunta:** Como o PIB per capita se modificou ao longo do período analisado?

**Visualização possível:** Gráfico de linhas ou comparação entre anos.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [23]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 3 — Crescimento populacional
**Pergunta:** Como a população total representada na base mudou ao longo do tempo?

**Visualização possível:** Tabela agregada ou gráfico de linhas.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [24]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 4 — Comparação entre continentes
**Pergunta:** Quais diferenças de expectativa de vida podem ser observadas entre continentes?

**Visualização possível:** Gráfico de linhas ou barras por continente.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [25]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 5 — PIB por continente
**Pergunta:** Como o PIB per capita se distribui entre os continentes em um ano escolhido?

**Visualização possível:** Gráfico de barras ou boxplot.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [26]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 6 — Expectativa de vida e PIB
**Pergunta:** Existe uma relação visual entre PIB per capita e expectativa de vida?

**Visualização possível:** Gráfico de dispersão.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [27]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 7 — Países com maior expectativa de vida
**Pergunta:** Quais países apresentam os maiores valores de expectativa de vida em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [28]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 8 — Países com menor expectativa de vida
**Pergunta:** Quais países apresentam os menores valores de expectativa de vida em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [29]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 9 — Maiores PIBs per capita
**Pergunta:** Quais países têm os maiores PIBs per capita em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [30]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 10 — Menores PIBs per capita
**Pergunta:** Quais países têm os menores PIBs per capita em um ano escolhido?

**Visualização possível:** Tabela de ranking ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [31]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 11 — Mudanças em um país
**Pergunta:** Como a expectativa de vida e o PIB per capita de um país escolhido mudaram ao longo do tempo?

**Visualização possível:** Gráfico de linhas com evolução temporal.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [32]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 12 — Disparidade entre países
**Pergunta:** A diferença entre países com maior e menor expectativa de vida mudou ao longo do período?

**Visualização possível:** Comparação de valores por ano.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [33]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 13 — Distribuição da expectativa de vida
**Pergunta:** Como os valores de expectativa de vida estão distribuídos em um ano escolhido?

**Visualização possível:** Histograma ou boxplot.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [34]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 14 — Distribuição do PIB per capita
**Pergunta:** Como os valores de PIB per capita estão distribuídos em um ano escolhido?

**Visualização possível:** Histograma ou boxplot.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [35]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 15 — População e expectativa de vida
**Pergunta:** É possível observar algum padrão entre tamanho da população e expectativa de vida?

**Visualização possível:** Gráfico de dispersão.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [36]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 16 — Variação entre continentes
**Pergunta:** Qual continente apresentou maior mudança na expectativa de vida entre dois anos escolhidos?

**Visualização possível:** Tabela comparativa ou gráfico de barras.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [37]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 17 — Variação do PIB por continente
**Pergunta:** Como o PIB per capita médio de cada continente variou entre dois anos?

**Visualização possível:** Gráfico de barras agrupadas ou linhas.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [38]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 18 — Comparação de anos
**Pergunta:** Quais diferenças gerais aparecem ao comparar o primeiro e o último ano da base?

**Visualização possível:** Tabela comparativa e gráficos.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [39]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 19 — Valores extremos
**Pergunta:** Existem países com valores muito diferentes dos demais em expectativa de vida ou PIB per capita?

**Visualização possível:** Boxplot e inspeção dos registros.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [40]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

### Análise 20 — Relação entre os três indicadores
**Pergunta:** Como expectativa de vida, PIB per capita e população se comportam em conjunto?

**Visualização possível:** Gráfico de dispersão; tamanho dos pontos pode representar população.

**Etapa atual: 🟩 ANÁLISE**

Registrem abaixo o código necessário para investigar a pergunta.

In [41]:
# Desenvolver a análise do grupo

**Interpretação do grupo:**

[Escrever aqui o que os resultados mostram, citando evidências do gráfico ou da tabela.]

## 6. Funções de Visualização e Análise Gráfica
**Etapa atual: 🟩 ORGANIZAÇÃO DO CÓDIGO**

Se houver gráficos repetidos, criem uma função simples para reaproveitar o código. Esta etapa é recomendada no enunciado, mas mantenham a solução compatível com o que foi visto em aula.

In [42]:
# Definir aqui uma função de visualização, se necessário

In [43]:
# Testar a função criada em uma das análises

## 7. Interpretação das Visualizações
**Etapa atual: 🟪 INTERPRETAÇÃO**

Reúnam as interpretações das análises escolhidas e expliquem:
- O que o gráfico ou a tabela mostra?
- Que comparação ou tendência chamou atenção?
- A pergunta inicial foi respondida?
- Existe alguma limitação que precisa ser considerada?

## 8. Síntese dos Principais Insights
**Etapa atual: 🟪 SÍNTESE**

Registrem os principais achados com base nos resultados produzidos pelo grupo.

1. **Insight 1:** [preencher com evidência da análise]
2. **Insight 2:** [preencher com evidência da análise]
3. **Insight 3:** [preencher com evidência da análise]

**Possíveis aplicações práticas:** que relação esses resultados podem ter com políticas públicas, saúde, condições econômicas ou desigualdades entre países?

## 9. Conclusão e Reflexão
**Etapa atual: 🟪 CONCLUSÃO**

- Quais foram os principais aprendizados?
- Quais foram as maiores dificuldades na coleta, limpeza ou análise?
- Quais limitações da base devem ser consideradas?
- O objetivo inicial foi atendido? Justifiquem com os resultados encontrados.

## 10. Referências
**Etapa atual: ⬜ FINALIZAÇÃO**

Registrem a fonte do dataset e quaisquer outras fontes utilizadas.

## Checklist antes da entrega
- [ ] As seis partes recomendadas no enunciado foram contempladas.
- [ ] Os valores ausentes foram removidos.
- [ ] Os registros duplicados foram eliminados.
- [ ] Foram realizadas pelo menos 15 análises.
- [ ] Cada visualização tem título, identificação dos eixos e interpretação.
- [ ] As decisões e observações estão documentadas em Markdown.
- [ ] O notebook foi executado do início ao fim e revisado.
- [ ] As análises e interpretações refletem o trabalho autoral do grupo.

**Atenção:** o enunciado proíbe o uso de IA generativa para geração automática de código ou análise. Utilizem este arquivo apenas como estrutura de organização e desenvolvam as soluções e interpretações por conta própria.